In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
path_main = Path("../../simulations")

path_data = path_main / "20251027"

path_save = path_main / "20251027_summary"
if not path_save.exists():
    path_save.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(search_type, cr, br, pm):
    folder = f"20251027_{search_type}_pmaxRepeat_canvasRadius{cr}_beamRadius{br}_pm{pm:.0e}_pfa{pm:.0e}"
    files = sorted(list((path_data / folder).glob("*_summary.csv")))

    data_list = []
    for f in files:
        data_list.append(pd.read_csv(f, index_col=0))
    df = pd.concat(data_list).reset_index(drop=True)
    return df

In [4]:
def load_ph(search_type, cr, br, pm):
    folder = f"20251027_{search_type}_pmaxRepeat_canvasRadius{cr}_beamRadius{br}_pm{pm:.0e}_pfa{pm:.0e}"
    files = sorted(list((path_data / folder).glob("*_ph.csv")))

    p_list = []
    h_list = []
    for f in files:
        df = pd.read_csv(f, index_col=0)
        p_list.append(df["p_max_all"])
        h_list.append(df["h_actual_all"])

    # Pad ragged list to array
    len_max = max(len(h) for h in h_list)
    h_actual_pad = np.array([h.tolist() + [np.nan] * (len_max - len(h)) for h in h_list])
    p_max_pad = np.array([p.tolist() + [np.nan] * (len_max - len(p)) for p in p_list])

    ds = xr.Dataset(
        data_vars=dict(
            h_actual=(["run", "ping"], h_actual_pad),
            p_max=(["run", "ping"], p_max_pad),
        ),
        coords=dict(
            run=("run", range(1, 1001)),
            ping=("ping", range(len_max)),
        ),
    )
    return ds 

In [5]:
cr_all = [5,8,10]
br_all = [1,2]
pm_all = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

In [6]:
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for pm in pm_all:
            df = load_summary(search_type, cr=cr, br=br, pm=pm)
            df.to_csv(path_save / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pm:.0e}_summary.csv")

In [7]:
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for pm in pm_all:
            df = load_ph(search_type, cr=cr, br=br, pm=pm)
            df.to_netcdf(path_save / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pm:.0e}_ph.nc")

In [8]:
search_type = "MAP"
for cr in cr_all:
    for br in br_all:
        for pm in pm_all:
            df = load_summary(search_type, cr=cr, br=br, pm=pm)
            df.to_csv(path_save / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pm:.0e}_summary.csv")

In [9]:
search_type = "MAP"
for cr in cr_all:
    for br in br_all:
        for pm in pm_all:
            df = load_ph(search_type, cr=cr, br=br, pm=pm)
            df.to_netcdf(path_save / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pm:.0e}_ph.nc")